In [2]:

import sys
!{sys.executable} -m pip install transformers datasets torch sentencepiece

  Using cached pyyaml-6.0.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 2.1 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━

Language cannot go directly into a model.

It must first become numbers — without losing meaning.

1. transformers

Main NLP library.

Think of it as:

Week 1:

sklearn

Week 3:

transformers

It gives:

BERT
GPT-style models
tokenizers
pretrained NLP models

without building from scratch.

2. datasets

Dataset manager.

Instead of manually downloading text datasets:

load_dataset(...)

Much easier.

Like torchvision.datasets for NLP.

3. torch

Same PyTorch.

Transformers are still neural networks.

Nothing new here.

4. sentencepiece

Tokenizer library.

You'll understand this soon.

It helps split text into model-readable pieces.


Transformer Revolution

This changed everything.

Core idea:

A word's meaning depends on surrounding words.

Not fixed.

Context matters.


--------

Attention — the core idea

We'll learn math later.

For now:

Attention means:

When understanding a word, look at other relevant words.

Example:

Sentence:

The animal didn't cross the street because it was tired.

What is:

it

?

Human:

Probably:

animal

because "tired" relates to animal.

Transformer learns this by attention.

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

sentence = "The bank can guarantee deposits will eventually cover future tuition costs."
tokens   = tokenizer.tokenize(sentence)
print("Tokens:", tokens)
print("Count: ", len(tokens))

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokens: ['the', 'bank', 'can', 'guarantee', 'deposits', 'will', 'eventually', 'cover', 'future', 'tuition', 'costs', '.']
Count:  12


In [4]:
encoding = tokenizer(
    sentence,
    return_tensors='pt',       # return PyTorch tensors
    padding=True,
    truncation=True,
    max_length=128
)

print("input_ids:      ", encoding['input_ids'])
print("attention_mask: ", encoding['attention_mask'])
print("Shape:          ", encoding['input_ids'].shape)

input_ids:       tensor([[  101,  1996,  2924,  2064, 11302, 10042,  2097,  2776,  3104,  2925,
         15413,  5366,  1012,   102]])
attention_mask:  tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
Shape:           torch.Size([1, 14])


In [5]:
ids    = encoding['input_ids'][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(ids)

for tok, id_ in zip(tokens, ids):
    print(f"  {tok:15s} {id_}")

  [CLS]           101
  the             1996
  bank            2924
  can             2064
  guarantee       11302
  deposits        10042
  will            2097
  eventually      2776
  cover           3104
  future          2925
  tuition         15413
  costs           5366
  .               1012
  [SEP]           102


In [6]:
sentences = [
    "I love this movie!",
    "This was a complete waste of time.",
    "Decent, nothing special."
]

batch = tokenizer(
    sentences,
    return_tensors='pt',
    padding=True,       # pad shorter sequences to match longest
    truncation=True,
    max_length=64
)

print("input_ids shape:      ", batch['input_ids'].shape)
print("attention_mask shape: ", batch['attention_mask'].shape)
print()
print("Padding tokens (0) in mask:")
print(batch['attention_mask'])

input_ids shape:       torch.Size([3, 10])
attention_mask shape:  torch.Size([3, 10])

Padding tokens (0) in mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0]])


Big Picture

This section teaches:

Language cannot directly enter BERT.

It becomes:

Sentence
↓
Tokens
↓
Special tokens ([CLS],[SEP])
↓
Token IDs
↓
Padding
↓
Attention mask
↓
Tensor
↓
BERT


Special Tokens

Very important.

BERT automatically adds these.

[CLS]

Appears at start.

Example:

[CLS] I love NLP [SEP]

Meaning:

Sentence summary token


[SEP]

End marker.

Separates sentences.

Example:

Sentence A [SEP] Sentence B

Meaning:

Boundary token

Like punctuation for BERT.

--------------

Padding — why needed

This is extremely important.

Suppose batch:

Sentence 1:

I love this

3 tokens.

Sentence 2:

This movie was unexpectedly emotional and brilliant

8 tokens.

Neural networks need:

same shape

Cannot have:

[3]
[8]

in one tensor.

So tokenizer pads shorter ones.

Example:

I love this

becomes:

I love this [PAD] [PAD] [PAD]

Attention mask tells:

Which tokens are real?

Example:

Sentence:

I love NLP [PAD] [PAD]

Mask:

1 1 1 0 0

Meaning:

1 = pay attention
0 = ignore

In [7]:
# Attention in plain English:
# For each word, ask: "how relevant is every other word to understanding me?"
# Words that are relevant get high attention weights (close to 1.0)
# Words that are irrelevant get low attention weights (close to 0.0)
# Your new representation = weighted sum of all other words' representations

# The "bank" example — same word, different contexts:
sentences = [
    "I deposited money at the bank.",      # bank = financial
    "We sat by the river bank at sunset."  # bank = geography
]
# In sentence 1, "bank" attends strongly to "deposited" and "money"
# In sentence 2, "bank" attends strongly to "river" and "sat"
# Same word → completely different embedding. This is what transformers do.

In [ ]:
# See attention weights with BertViz (optional, visual)
import sys
!{sys.executable} -m pip install bertviz

from transformers import BertModel, BertTokenizer
from bertviz import head_view

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model     = BertModel.from_pretrained('bert-base-uncased', output_attentions=True)

sentence  = "The animal didn't cross the street because it was too tired"
inputs    = tokenizer.encode(sentence, return_tensors='pt')
outputs   = model(inputs)
attention = outputs.attentions   # tuple of attention weights per layer
tokens    = tokenizer.convert_ids_to_tokens(inputs[0])

head_view(attention, tokens)   # renders an interactive attention visualisation

The key insight

Old NLP:

One word = one fixed meaning

Transformer:

One word = meaning depends on surrounding words

This is called contextual understanding.

That is why BERT is powerful.

What are attention weights?

Attention gives scores.

Think:

0 → ignore
1 → strong focus

Example of bank
| Word      | Attention |
| --------- | --------: |
| money     |       0.9 |
| deposited |       0.8 |
| at        |       0.1 |
| I         |       0.1 |


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model     = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

sentences = [
    "I love this film.",
    "I hate this film.",
    "The cat sat on the mat.",
]

with torch.no_grad():
    inputs  = tokenizer(sentences, return_tensors='pt',
                        padding=True, truncation=True, max_length=64)
    outputs = model(**inputs)

# outputs.last_hidden_state shape: [batch, seq_len, 768]
print("Last hidden state shape:", outputs.last_hidden_state.shape)

# CLS token = index 0 — represents the whole sentence
cls_embeddings = outputs.last_hidden_state[:, 0, :]
print("CLS embedding shape:", cls_embeddings.shape)   # [3, 768]

In [ ]:
from torch.nn.functional import cosine_similarity

e0 = cls_embeddings[0]   # "I love this film"
e1 = cls_embeddings[1]   # "I hate this film"
e2 = cls_embeddings[2]   # "The cat sat on the mat"

sim_01 = cosine_similarity(e0.unsqueeze(0), e1.unsqueeze(0)).item()
sim_02 = cosine_similarity(e0.unsqueeze(0), e2.unsqueeze(0)).item()

print(f"'love film' vs 'hate film': {sim_01:.3f}")   # high — same structure
print(f"'love film' vs 'cat mat':   {sim_02:.3f}")   # low — different topic

Big picture — what are we doing here?

So far:

Text → Tokens → Attention

Now next step:

BERT converts the whole sentence into numerical meaning representations called embeddings.

BERT builds a meaning map.

Example:

Close together:

"I love this film"
"This movie is amazing"

Far apart:

"I love this film"
"The cat sat on the mat"

So:

Embedding = location of meaning in a high-dimensional map.

BERT does not directly say:

positive,
negative,
spam,
intent,

Instead it says:

"Here is a rich numerical representation of what this text means."

That representation is called an embedding.


-------------

This is why:

Raw BERT ≠ sentiment classifier.

BERT gives:

meaning representation

Text
↓
BERT
↓
CLS embedding
↓
Classifier head
↓
Sentiment / Intent / Spam / etc


One-line memory shortcut

Remember this:

BERT converts sentences into embeddings — dense meaning vectors. The CLS embedding acts as a summary of the whole sentence and becomes the input for NLP classifiers.